# ImageEval 2026 — Task 1b (Hallucination Detection) · Colab baseline

End-to-end baseline for **Task 1b**: given an **image** and **three statements**
(exactly one grounded), label **each** statement **True** or **False**.

**Model.** [`Qwen2.5-VL-3B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct),
a strong, current vision-language model, judges each statement against the image; outputs
are parsed with the **official** `evaluate_tf` parser. It fits a **free Colab T4 (16 GB)**.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` (GPU, **not** TPU).

**Output:** `predictions_<lang>.csv` (`id,statement_index,raw_prediction,prediction_parsed`)
and a Codabench-ready `prediction_<lang>.zip`. Switch tracks with `LANG = "en"`/`"msa"`.
Defaults to `devtest` (blind, for submission); set `SPLIT="dev"` to get a local score.

## 1. Install dependencies

Installs once, then **auto-restarts the runtime** so the freshly installed versions load
cleanly (this avoids `PIL`/`transformers` half-upgrade import errors). The restart only
happens on the **first** run — when it does, just press **Run all** again to continue.

In [ ]:
# Install once, then restart so new versions load cleanly. Re-run "Run all" after the restart.
import os
_FLAG = "/content/.imageeval_deps_1b"
if not os.path.exists(_FLAG):
    !pip install -q -U "transformers>=4.49.0" accelerate bitsandbytes qwen-vl-utils "huggingface_hub[hf_transfer]"
    open(_FLAG, "w").close()
    print("Dependencies installed — restarting runtime. Press 'Run all' again to continue.")
    import IPython; IPython.Application.instance().kernel.do_shutdown(True)

## 2. Configuration

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less GPU fragmentation

REPO_ID   = "QCRI/ImageEval2026-Task1-AynVQA"
TASK      = "task1b"
LANG      = "en"        # "en" or "msa"  -> output is predictions_<LANG>.csv
SPLIT     = "devtest"   # "devtest"/"test" -> blind (submit) | "dev"/"train" -> labelled (scored)
MAX_ITEMS = None        # e.g. 20 for a quick smoke test; None = whole split

VLM_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"   # 7B variant: "Qwen/Qwen2.5-VL-7B-Instruct"
QUANTIZE  = True       # 4-bit (bitsandbytes) -> fits a T4 comfortably; set False on L4/A100
print(f"config: {TASK}_{LANG} / {SPLIT}  (quantized: {QUANTIZE})")

## 3. Download the split + images from the Hub

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(REPO_ID, filename=f"{TASK}/{SPLIT}_{LANG}.jsonl", repo_type="dataset")
records = [json.loads(l) for l in open(jsonl, encoding="utf-8") if l.strip()]
if MAX_ITEMS:
    records = records[:MAX_ITEMS]
print(len(records), "items;  labelled:", "labels" in records[0])

def fetch(rel):
    return hf_hub_download(REPO_ID, filename=rel, repo_type="dataset")

needed = sorted({r["image"] for r in records})
paths = {}
with ThreadPoolExecutor(max_workers=16) as ex:
    for rel, p in tqdm(zip(needed, ex.map(fetch, needed)), total=len(needed), desc="images"):
        paths[rel] = p

## 4. Load the model

Uses bf16 on Ampere+ (`L4`/`A100`), fp16 on a T4.

In [ ]:
import torch
from transformers import (Qwen2_5_VLForConditionalGeneration,
                          AutoProcessor, BitsAndBytesConfig)
from qwen_vl_utils import process_vision_info

# Fail fast on a CPU runtime: this model is unusably slow without a GPU.
assert torch.cuda.is_available(), \
    "No GPU detected. Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

QUANTIZE = globals().get("QUANTIZE", True)   # falls back to True if cell 2 was skipped
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype:", dtype, "| quantized:", QUANTIZE)

# 4-bit NF4 quantization (~halves weight memory) so the model fits a 16 GB T4.
quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
) if QUANTIZE else None

# Cap image resolution -> bounds vision tokens (attention memory grows with seq-len^2).
# 512*28*28 keeps a 16 GB T4 happy; raise toward 1280*28*28 on an L4/A100 for more detail.
MAX_PIXELS = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype, device_map="auto", quantization_config=quant).eval()

## 5. Official True/False parser

Verbatim copy of `backbone.evaluate_tf` — the same parser the Codabench 1b scorer uses,
so the score printed below matches the leaderboard.

In [ ]:
# --- True/False parser: verbatim copy of the official backbone.evaluate_tf ---
# This is what the Codabench 1b scorer uses, so local scores match exactly.
import re
from dataclasses import dataclass
from typing import Optional

TRUE_TOKENS = [r"\btrue\b", r"\byes\b", r"\bصح\b", r"\bصحيح\b", r"\bصحيحة\b",
               r"\bالصحيح\b", r"\bالخطأ\b"]
FALSE_TOKENS = [r"\bfalse\b", r"\bfalsy\b", r"\bno\b", r"\bخطأ\b", r"\bالخطأ\b",
                r"\bغلط\b", r"\bغير\s+صحيح(?:ة)?\b", r"\bغير\s+صحيحة\b",
                r"\bخاطئ\b", r"\bخاطئة\b"]
ABSTAIN_PATTERNS = [
    r"\b(can(?:not|'t)\s+determine|can(?:not|'t)\s+tell|not\s+enough\s+information|cannot\s+be\s+sure|unclear)\b",
    r"لا\s+يمكن(?:نا)?\s+الجزم", r"لا\s+يمكن\s+الجزم",
    r"لا\s+يمكن\s+تحديد.*(?:صحة|خطأ|صحيح|خاطئ|العبارة)",
    r"لا\s+نستطيع\s+التأكد", r"لا\s+يمكن\s+الحكم"]
STRONG_CUES = [
    r"therefore[,:\s]*", r"final\s+answer[,:\s]*", r"the\s+answer\s+is[,:\s]*",
    r"correct\s+answer\s+is[,:\s]*", r"so\s+the\s+answer\s+is[,:\s]*",
    r"conclusion[,:\s]*", r"verdict[,:\s]*", r"determination[,:\s]*",
    r"final[,:\s]*(?:answer)?[,:\s]*", r"the\s+statement\s+is\s*[:\-–—,]?\s*",
    r"الإجابة\s+الصحيحة\s*(?:هي)?\s*[:：]?\s*",
    r"الجواب\s+الصحيح\s*(?:هو|هي)?\s*[:：]?\s*",
    r"الإجابة\s*(?:هي)?\s*[:：]?\s*", r"الجواب\s*(?:هو|هي)?\s*[:：]?\s*",
    r"إذًا\s*(?:الجواب|الجواب\s+هو|الإجابة|الإجابة\s+هي)?\s*[:：]?\s*",
    r"الإجابة\s+النهائية\s*(?:هي)?\s*[:：]?\s*"]


@dataclass
class EvalResult:
    pred: Optional[str]
    confidence: float
    needs_review: bool
    reason: str
    conflict: bool


def _normalize(text):
    t = (text or "").strip().replace("‏", "").replace("‎", "").lower()
    return re.sub(r"[ \t]+", " ", t)


def _strip_code_and_quotes(text):
    t = text or ""
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)
    t = re.sub(r"\".*?\"", " ", t, flags=re.DOTALL)
    t = re.sub(r"“.*?”", " ", t, flags=re.DOTALL)
    return t


def _match_label(fragment):
    for pat in TRUE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "true"
    for pat in FALSE_TOKENS:
        if re.search(pat, fragment, flags=re.IGNORECASE):
            return "false"
    return None


def _has_any(patterns, text):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def evaluate_tf(response):
    raw = response or ""
    if not raw.strip():
        return EvalResult(None, 0.0, True, "empty_response", conflict=False)
    text_noquotes = _normalize(_strip_code_and_quotes(raw))

    first_label_pos = first_label_value = None
    for pat, lab in [(p, "true") for p in TRUE_TOKENS] + [(p, "false") for p in FALSE_TOKENS]:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_label_pos is None or m.start() < first_label_pos):
            first_label_pos, first_label_value = m.start(), lab
    first_abstain_pos = None
    for pat in ABSTAIN_PATTERNS:
        m = re.search(pat, text_noquotes, flags=re.IGNORECASE)
        if m and (first_abstain_pos is None or m.start() < first_abstain_pos):
            first_abstain_pos = m.start()
    if first_label_pos is not None or first_abstain_pos is not None:
        if first_abstain_pos is not None and (first_label_pos is None or first_abstain_pos < first_label_pos):
            return EvalResult(None, 0.0, True, "abstain_before_label", conflict=False)
        if first_label_pos is not None and (first_abstain_pos is None or first_label_pos < first_abstain_pos):
            return EvalResult(first_label_value, 0.95, False, "first_explicit_label", conflict=False)

    best = None
    m0 = re.match(r"^\s*[\*\s_`]*((?:true|false)|(?:صح|صحيح|صحيحة)|(?:خطأ|غلط|خاطئ|خاطئة))\b",
                  text_noquotes, flags=re.IGNORECASE)
    if m0:
        label = _match_label(m0.group(1))
        if label:
            best = (3.0, label, "leading_label")
    for cue in STRONG_CUES:
        for m in re.finditer(cue, text_noquotes, flags=re.IGNORECASE):
            label = _match_label(text_noquotes[m.end():m.end() + 140])
            if label:
                cand = (3.0, label, "strong_cue")
                best = max(best, cand, key=lambda x: x[0]) if best else cand
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    standalone = [
        (re.compile(r"^[\*\s_`]*true\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "true"),
        (re.compile(r"^[\*\s_`]*false\s*[\.!\?]*[\*\s_`]*$", re.IGNORECASE), "false"),
        (re.compile(r"^[\*\s_`]*صح\s*[\.!\?]*[\*\s_`]*$"), "true"),
        (re.compile(r"^[\*\s_`]*(خطأ|غلط|خاطئ|خاطئة)\s*[\.!\?]*[\*\s_`]*$"), "false")]
    if not best or best[0] < 3.0:
        for ln in reversed(lines[-25:]):
            ln_norm = _normalize(ln)
            for rgx, lab in standalone:
                if rgx.match(ln_norm):
                    best = (2.0, lab, "standalone_label_line")
                    break
            if best:
                break
    if not best:
        tail = text_noquotes[-450:]
        matches = []
        for pat in TRUE_TOKENS:
            matches += [(mm.start(), "true") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        for pat in FALSE_TOKENS:
            matches += [(mm.start(), "false") for mm in re.finditer(pat, tail, flags=re.IGNORECASE)]
        if matches:
            matches.sort(key=lambda x: x[0])
            best = (1.0, matches[-1][1], "last_occurrence_in_tail")
    if not best:
        return EvalResult(None, 0.0, True, "no_label_found", conflict=False)
    score, label, reason = best
    conflict = _has_any(TRUE_TOKENS, text_noquotes) and _has_any(FALSE_TOKENS, text_noquotes)
    confidence = {3.0: 0.95, 2.0: 0.80, 1.0: 0.60}.get(score, 0.50)
    return EvalResult(label, confidence, conflict and score <= 1.0, reason, conflict)

## 6. Inference helpers

In [ ]:
PROMPT = (
    "You are checking a statement against an image for visual hallucination. "
    "Look only at what the image actually shows.\n\n"
    "Statement: \"{s}\"\n\n"
    "If the image clearly supports the statement, answer True. If the statement describes "
    "something that is not in the image or is contradicted by it (a hallucination), answer "
    "False. Answer with only one word: True or False."
)

@torch.no_grad()
def ask_vlm(image_path, text):
    conv = [{"role": "user", "content": [
        {"type": "image", "image": image_path},
        {"type": "text",  "text":  text}]}]
    prompt = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(conv)
    inputs = processor(text=[prompt], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    gen = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    out = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    del inputs, gen
    torch.cuda.empty_cache()          # release per-item activations -> avoids slow OOM
    return out

## 7. Run inference

One True/False judgement per statement (3 per item). We keep the model's **raw** text and the
**parsed** label (`evaluate_tf`). When the reply has no clear verdict we fall back to `false`
— and since 2 of every 3 statements really are false, that's the correct prior, not a freebie.
We report how often the fallback was used; inspect `raw_prediction` for real model behaviour.

In [ ]:
DEFAULT_LABEL = "false"   # fallback when evaluate_tf cannot read a verdict

rows = []   # (id, statement_index, raw_prediction, prediction_parsed)
n_fallback = 0
for r in tqdm(records, desc="infer"):
    for si, stmt in enumerate(r["statements"]):
        raw = ask_vlm(paths[r["image"]], PROMPT.format(s=stmt))
        parsed = evaluate_tf(raw).pred
        n_fallback += (parsed is None)
        rows.append((r["id"], si, raw, parsed or DEFAULT_LABEL))

print(f"done: {len(rows)} judgements  |  unparseable -> defaulted to {DEFAULT_LABEL!r}: {n_fallback}")

## 8. Write `predictions_<lang>.csv`

Columns: `id, statement_index, raw_prediction, prediction_parsed`. Switching `LANG` writes a
separate file (`predictions_en.csv` / `predictions_msa.csv`).

In [ ]:
import csv
OUT_CSV = f"predictions_{LANG}.csv"
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "statement_index", "raw_prediction", "prediction_parsed"])
    w.writerows(rows)
print("wrote", OUT_CSV, ":", len(rows), "rows")

## 9. Score (official metric: **combined accuracy**)

**Combined accuracy** (the winning metric) = fraction of items where *all three*
statements are correct. Also reports Q+ (true-statement) and Q− (false-statement)
accuracies and the hallucination rate — identical to the Codabench scorer.
Runs only when the split is labelled.

In [ ]:
gold = {r["id"]: r["labels"].index(True) for r in records if "labels" in r}
if gold:
    by_item = {}
    for iid, si, _raw, parsed in rows:
        by_item.setdefault(iid, {})[si] = parsed

    total = q_plus = q_minus = q_minus_total = combined = 0
    for iid, true_idx in gold.items():
        total += 1
        q_minus_total += 2
        pr = by_item.get(iid, {})
        labels = {i: evaluate_tf(pr.get(i, "")).pred for i in range(3)}
        ok_t = labels.get(true_idx) == "true"
        ok_f = [labels.get(i) == "false" for i in range(3) if i != true_idx]
        if ok_t:
            q_plus += 1
        q_minus += sum(ok_f)
        if ok_t and all(ok_f):
            combined += 1

    combined_acc = combined / total
    q_plus_acc   = q_plus / total
    q_minus_acc  = q_minus / q_minus_total
    print(f"split: {SPLIT}  ({total} items)")
    print(f"combined_accuracy : {combined_acc:.4f}   <- winning metric")
    print(f"q_plus_accuracy   : {q_plus_acc:.4f}")
    print(f"q_minus_accuracy  : {q_minus_acc:.4f}")
    print(f"hallucination_rate: {q_plus_acc - combined_acc:.4f}")
else:
    print(f"'{SPLIT}' is blind (no labels) — submit to Codabench to get the score.")

## 10. Build the Codabench submission

The leaderboard expects a zip with a `prediction.csv` of exactly
`id,statement_index,prediction` (the **parsed** label). Built from `prediction_parsed`;
the raw column stays local for auditing. Submit `prediction_<lang>.zip` to:
[task1b_en](https://www.codabench.org/competitions/17022/) ·
[task1b_msa](https://www.codabench.org/competitions/17021/).

In [ ]:
import zipfile
with open("prediction.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "statement_index", "prediction"])
    for iid, si, _raw, parsed in rows:
        w.writerow([iid, si, parsed])
zip_name = f"prediction_{LANG}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction.csv", "prediction.csv")
print("wrote", zip_name, " -> submit this file to Codabench")